# MSM Notebook 3 — STATE VECTOR LAYER

**전제:** Notebook 2 완료 (active universe + OHLCV/Flow/Foreign + macro 수집됨)

**확정 셋 (Q4=A, Q5=C):**
```
S_t = [r, v, f, σ, ℓ]

r = log(close_t / close_{t-1})              # adjusted close (매뉴얼 §9.3)
v = log(trade_value_t / 60d_mean)            # raw trade_value (매뉴얼 §9.4)
f = (foreign_5d + inst_5d) / trade_value_5d  # 금액 그대로 (Notebook 1 발견)
σ = std(r) over 20d                          # adjusted return
ℓ = log10(trade_value_60d / 100억)           # raw trade_value
```

**Dual normalization (Q5=C):**
- Cross-sectional z-score (시점별, 종목 간)
- Time-series z-score (종목별, 시간 축)

**§9 매뉴얼 함정 준수:**
- §9.3: M축은 adjusted close (분할 오염 방지)
- §9.4: trade_value는 raw OHLCV에만 있음
- §9.5: code = str.zfill(6) 강제
- §9.6: NaN 보간 금지 (상장 전 / lookback 부족은 NaN 유지)
- §9.8: CA flag (분할/IPO/병합) 적용 — 형 결정 필요

**산출:** `data/state/state_vector.parquet`

---
## 0. Setup + 보유 데이터 검증

In [ ]:
import os, json, glob
from pathlib import Path
import pandas as pd
import numpy as np
from google.colab import userdata

%cd /content/choonsimi-msm

ROOT = Path("data")
(ROOT / "state").mkdir(parents=True, exist_ok=True)

GH_TOKEN = userdata.get("GH_TOKEN")
REPORT = {}

# 보유 데이터 점검 (§1 추측 금지)
expected = {
    "delisted_ohlcv": ROOT / "raw" / "delisted" / "ohlcv",
    "delisted_flow":  ROOT / "raw" / "delisted" / "flow",
    "active_ohlcv_adj": ROOT / "raw" / "active" / "ohlcv_adj",
    "active_ohlcv":     ROOT / "raw" / "active" / "ohlcv",
    "active_flow":      ROOT / "raw" / "active" / "flow",
    "active_foreign":   ROOT / "raw" / "active" / "foreign",
    "universe":         ROOT / "universe" / "universe_msm.csv",
    "macro":            ROOT / "raw" / "macro" / "macro_10y.json",
}
for k, p in expected.items():
    if p.is_dir():
        files = sorted(glob.glob(str(p / "*.parquet")))
        print(f"{k}: dir, {len(files)} files")
    else:
        print(f"{k}: {'✓' if p.exists() else '❌ MISSING'} ({p})")

print("\n위 결과에서 ❌ 있으면 Notebook 2 먼저 실행 완료해야 함.")

---
## Step A — 데이터 통합 (delisted + active)

**원칙:**
- adjusted close: 모멘텀(r, σ) 계산용
- raw trade_value: volume(v), liquidity(ℓ) 계산용
- Flow: 금액 단위(원). 추가 환산 불필요.

**컬럼 표준:** code (str, zfill6), date (datetime)

In [ ]:
def load_parquet_dir(path):
    files = sorted(glob.glob(str(path / "*.parquet")))
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    if "code" in df.columns:
        df["code"] = df["code"].astype(str).str.zfill(6)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    return df

# A-1. OHLCV adjusted 통합 (delisted + active)
df_adj_del = load_parquet_dir(ROOT / "raw" / "delisted" / "ohlcv")
df_adj_act = load_parquet_dir(ROOT / "raw" / "active" / "ohlcv_adj")
df_adj_del["source"] = "delisted"
df_adj_act["source"] = "active"
df_adj = pd.concat([df_adj_del, df_adj_act], ignore_index=True)
df_adj = df_adj.drop_duplicates(subset=["code", "date"], keep="last")
print(f"OHLCV adj: {len(df_adj)}행, {df_adj['code'].nunique()}종목, {df_adj['date'].min().date()} ~ {df_adj['date'].max().date()}")

In [ ]:
# A-2. OHLCV raw 통합 — trade_value, market_cap 컬럼 보유
# delisted는 active OHLCV raw 없음 → delisted는 추후 거래대금 보강 필요
df_raw_act = load_parquet_dir(ROOT / "raw" / "active" / "ohlcv")
print(f"Active OHLCV raw: {len(df_raw_act)}행, {df_raw_act['code'].nunique()}종목")
print(f"컬럼: {list(df_raw_act.columns)}")

# delisted: 거래대금 컬럼 없음 (Notebook 1 수집은 adjusted만)
# 매뉴얼 §9.4 — adjusted엔 trade_value 없음. delisted v, ℓ 계산을 위해선 raw 별도 수집 필요.
# 임시: delisted는 raw_close * volume 근사. 후속 단계에서 raw 재수집 결정.
print("\n⚠ delisted raw trade_value 없음 — adjusted close × volume 근사 사용")
df_adj_del["trade_value_approx"] = df_adj_del["close"] * df_adj_del["volume"]

In [ ]:
# A-3. Flow 통합
df_flow_del = load_parquet_dir(ROOT / "raw" / "delisted" / "flow")
df_flow_act = load_parquet_dir(ROOT / "raw" / "active" / "flow")
df_flow_del["source"] = "delisted"
df_flow_act["source"] = "active"
df_flow = pd.concat([df_flow_del, df_flow_act], ignore_index=True)
df_flow = df_flow.drop_duplicates(subset=["code", "date"], keep="last")
print(f"Flow: {len(df_flow)}행, {df_flow['code'].nunique()}종목")
print(f"컬럼: {list(df_flow.columns)}")

# 제로섬 검산 (매뉴얼 §10)
flow_cols = ["기관합계", "기타법인", "개인", "외국인합계"]
if all(c in df_flow.columns for c in flow_cols):
    zerosum = (df_flow[flow_cols].sum(axis=1)).abs().max()
    print(f"제로섬 검산: max|sum|={zerosum:.0f} (≈0이어야 정상)")

In [ ]:
# A-4. Foreign 지분율 (active만 — delisted는 별도 미수집)
df_foreign = load_parquet_dir(ROOT / "raw" / "active" / "foreign")
print(f"Foreign: {len(df_foreign)}행, {df_foreign['code'].nunique()}종목")
print(f"컬럼: {list(df_foreign.columns)}")
# 매뉴얼 §5.2 — 컬럼명: '한도소진률(%)', '지분율(%)' 등. 첫 셀에서 확인 필요.

---
## Step B — State Vector 5축 계산

**기준 패널:** (code, date) — adjusted OHLCV 인덱스
**계산 순서:** r → σ → v → ℓ → f

In [ ]:
# B-1. r (log return)
# 매뉴얼 §9.3 — adjusted close 사용
df_adj = df_adj.sort_values(["code", "date"]).reset_index(drop=True)
df_adj["r"] = df_adj.groupby("code")["close"].transform(lambda x: np.log(x / x.shift(1)))
print(f"r 계산 완료. NaN 비율: {df_adj['r'].isna().mean()*100:.2f}%")

In [ ]:
# B-2. σ (20일 realized volatility)
df_adj["sigma"] = df_adj.groupby("code")["r"].transform(lambda x: x.rolling(20, min_periods=20).std())
print(f"σ 계산 완료. NaN 비율: {df_adj['sigma'].isna().mean()*100:.2f}%")

In [ ]:
# B-3. v (log volume ratio) + ℓ (log liquidity)
# raw trade_value 우선, delisted는 근사값 사용
# active와 delisted 구분 후 결합

# Active: raw trade_value 사용
df_v_act = df_raw_act[["code", "date", "trade_value"]].copy()

# Delisted: 근사 (adjusted close × volume)
df_v_del = df_adj_del[["code", "date"]].copy()
df_v_del["trade_value"] = df_adj_del["trade_value_approx"]

df_v = pd.concat([df_v_act, df_v_del], ignore_index=True).drop_duplicates(subset=["code","date"], keep="last")
df_v = df_v.sort_values(["code", "date"]).reset_index(drop=True)

# 60일 평균
df_v["trade_value_60d"] = df_v.groupby("code")["trade_value"].transform(
    lambda x: x.rolling(60, min_periods=60).mean()
)
df_v["v"] = np.log(df_v["trade_value"] / df_v["trade_value_60d"])
df_v["ell"] = np.log10(df_v["trade_value_60d"] / 1e10)  # /100억

print(f"v / ℓ 계산 완료. v NaN: {df_v['v'].isna().mean()*100:.2f}%, ℓ NaN: {df_v['ell'].isna().mean()*100:.2f}%")

In [ ]:
# B-4. f (flow imbalance) — 5일 합 / 5일 거래대금 합
# Flow 단위: 원 (Notebook 1 확인). raw_close 곱하기 불필요.

df_flow = df_flow.sort_values(["code", "date"]).reset_index(drop=True)
for col in ["외국인합계", "기관합계"]:
    if col in df_flow.columns:
        df_flow[f"{col}_5d"] = df_flow.groupby("code")[col].transform(
            lambda x: x.rolling(5, min_periods=5).sum()
        )

# trade_value 5일 합 — df_v에서 가져옴
df_v["trade_value_5d"] = df_v.groupby("code")["trade_value"].transform(
    lambda x: x.rolling(5, min_periods=5).sum()
)

# Flow + trade_value 5일 합 merge
df_f = df_flow[["code", "date", "외국인합계_5d", "기관합계_5d"]].merge(
    df_v[["code", "date", "trade_value_5d"]], on=["code", "date"], how="inner"
)
df_f["f"] = (df_f["외국인합계_5d"] + df_f["기관합계_5d"]) / df_f["trade_value_5d"]
print(f"f 계산 완료. NaN: {df_f['f'].isna().mean()*100:.2f}%")

In [ ]:
# B-5. 5축 통합 패널
panel = df_adj[["code", "date", "r", "sigma"]].merge(
    df_v[["code", "date", "v", "ell"]], on=["code", "date"], how="left"
).merge(
    df_f[["code", "date", "f"]], on=["code", "date"], how="left"
)

panel = panel.rename(columns={"sigma": "σ", "ell": "ℓ"})
# 컬럼명 ASCII로
panel.columns = ["code", "date", "r", "sigma", "v", "ell", "f"]
panel = panel[["code", "date", "r", "v", "f", "sigma", "ell"]]

print(f"State Panel: {len(panel)}행, {panel['code'].nunique()}종목")
print(f"NaN 비율:")
for c in ["r", "v", "f", "sigma", "ell"]:
    print(f"  {c}: {panel[c].isna().mean()*100:.2f}%")

print("\n샘플:")
print(panel.head(3))
print(panel.dropna().head(3))

---
## Step C — Dual Normalization

**Cross-sectional (CS):** 매 date별로 모든 종목 간 z-score
**Time-series (TS):** 매 code별로 시간 축 z-score

둘 다 저장. 다음 단계에서 비교.

In [ ]:
# C-1. Cross-sectional z-score (시점별)
axes = ["r", "v", "f", "sigma", "ell"]
for ax in axes:
    panel[f"{ax}_cs"] = panel.groupby("date")[ax].transform(
        lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan
    )
print("Cross-sectional z-score 완료.")

In [ ]:
# C-2. Time-series z-score (종목별)
for ax in axes:
    panel[f"{ax}_ts"] = panel.groupby("code")[ax].transform(
        lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan
    )
print("Time-series z-score 완료.")

print(f"\n최종 panel: {len(panel)}행, {len(panel.columns)}컬럼")
print(f"컬럼: {list(panel.columns)}")

---
## Step D — 저장 + 검증

In [ ]:
# D-1. 저장
out = ROOT / "state" / "state_vector.parquet"
panel.to_parquet(out, index=False)
print(f"✓ 저장: {out}")
print(f"  행: {len(panel)}, 종목: {panel['code'].nunique()}")
print(f"  기간: {panel['date'].min().date()} ~ {panel['date'].max().date()}")

In [ ]:
# D-2. 검증
validation = {
    "total_rows": int(len(panel)),
    "unique_codes": int(panel["code"].nunique()),
    "date_min": str(panel["date"].min().date()),
    "date_max": str(panel["date"].max().date()),
    "nan_pct": {ax: float(panel[ax].isna().mean()*100) for ax in axes},
    "nan_pct_cs": {f"{ax}_cs": float(panel[f'{ax}_cs'].isna().mean()*100) for ax in axes},
    "nan_pct_ts": {f"{ax}_ts": float(panel[f'{ax}_ts'].isna().mean()*100) for ax in axes},
    "stats": {ax: {
        "min": float(panel[ax].min()),
        "max": float(panel[ax].max()),
        "mean": float(panel[ax].mean()),
        "std": float(panel[ax].std()),
    } for ax in axes},
}

with open(ROOT / "checks" / "state_vector_validation.json", "w", encoding="utf-8-sig") as f:
    json.dump(validation, f, ensure_ascii=False, indent=2, default=str)

print(json.dumps(validation, ensure_ascii=False, indent=2, default=str))

---
## Step E — Commit + Push

In [ ]:
import subprocess
origin_url = subprocess.run(["git", "remote", "get-url", "origin"], capture_output=True, text=True).stdout.strip()
auth_url = origin_url.replace("https://", f"https://{GH_TOKEN}@")

!git config user.email "msm@stanleyim.local"
!git config user.name "stanleyim"
!git remote set-url origin {auth_url}
!git add data/state/ data/checks/state_vector_validation.json msm_state_vector.ipynb
!git commit -m "Notebook 3: State vector build with dual normalization"
!git push origin main
!git remote set-url origin {origin_url}
!git log --oneline -3

---
## 다음 단계: Notebook 4 — Regime Detection (MSM STEP 3)

- state_vector.parquet 입력
- regime: trend / range / shock / transition
- 방법: volatility + flow + return structure clustering